In [2]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm

# 如果是 Apple Silicon (M1/M2/M3)，优先使用 mps 加速
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

model_id = "EleutherAI/pythia-70m-deduped"
model = AutoModelForCausalLM.from_pretrained(
  model_id,
  revision="step3000",
  cache_dir="./pythia-70m-deduped/step3000",
).to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(
  model_id,
  revision="step3000",
  cache_dir="./pythia-70m-deduped/step3000",
)
tokenizer.pad_token = tokenizer.eos_token

/opt/anaconda3/envs/streaming/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/opt/anaconda3/envs/streaming/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


/opt/anaconda3/envs/streaming/lib/python3.8/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/opt/anaconda3/envs/streaming/lib/python3.8/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/opt/anaconda3/envs/streaming/lib/python3.8/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
Some weights of GPTNeoXForCausalLM were not initialized from the model checkpoint at EleutherAI/pythia-70m-deduped and are newly initialized: ['gpt_n

In [3]:
def measure_latency(model, tokenizer, prompt, max_new_tokens=64):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # ===== 测试 TTFT (第一个Token的时间) =====
    start_time = time.perf_counter()
    with torch.no_grad():
        outputs = model(input_ids=inputs.input_ids, use_cache=True)
        next_token_logits = outputs.logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)
    
    first_token_time = time.perf_counter()
    ttft = first_token_time - start_time
    
    # ===== 测试 TPOT (后续生成每个Token的平均时间) =====
    past_key_values = outputs.past_key_values
    input_ids = next_token
    
    start_tpot = time.perf_counter()
    for _ in range(max_new_tokens - 1):
        with torch.no_grad():
            out = model(input_ids=input_ids, past_key_values=past_key_values, use_cache=True)
            past_key_values = out.past_key_values
            next_token_logits = out.logits[:, -1, :]
            input_ids = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)
            
    end_time = time.perf_counter()
    
    tpot_total_time = end_time - start_tpot
    tpot = tpot_total_time / (max_new_tokens - 1)
    
    total_time = end_time - start_time
    throughput = max_new_tokens / total_time 
    
    return {
        "TTFT (ms)": round(ttft * 1000, 2),
        "TPOT (ms/token)": round(tpot * 1000, 2),
        "Throughput (tokens/s)": round(throughput, 2)
    }

prompt = "The future of artificial intelligence in natural language processing is"
latency_metrics = measure_latency(model, tokenizer, prompt, max_new_tokens=64)
print("=== 性能指标 Baseline ===")
for k, v in latency_metrics.items():
    print(f"{k}: {v}")

=== 性能指标 Baseline ===
TTFT (ms): 2598.11
TPOT (ms/token): 94.44
Throughput (tokens/s): 7.49


In [4]:
def calculate_ppl(model, tokenizer, dataset, text_column, max_length=1024, sample_size=10, join_texts=False):
    """
    计算数据集的 PPL（困惑度）。
    join_texts: 针对 Wikitext 这种一行很短的数据集，设为 True 会先拼接再评测，避免被短行阶段；
                针对 PG-19 这种单本文档非常长的数据集，设为 False 单独处理。
    """
    total_nll = 0.0
    total_tokens = 0
    
    # 提取有效文本
    texts = [dataset[i][text_column] for i in range(min(sample_size, len(dataset))) if dataset[i][text_column].strip()]
    
    if join_texts:
        texts = ["\n\n".join(texts)]
        
    for text in tqdm(texts, desc="Evaluating PPL"):
        # 取出每个文本（或者拼接后的长文本）
        encodings = tokenizer(text, return_tensors="pt")
        seq_len = encodings.input_ids.size(1)

        # 按照 max_length 切块，stride = max_length（无重叠切块，最快）
        for begin_loc in range(0, seq_len, max_length):
            end_loc = min(begin_loc + max_length, seq_len)
            trg_len = end_loc - begin_loc
            
            # Causal LM 的 Loss 至少需要 2 个 token
            if trg_len < 2:
                break
                
            input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
            target_ids = input_ids.clone()
            
            with torch.no_grad():
                outputs = model(input_ids, labels=target_ids)
                # target_ids 中有效的 token 数量其实是 trg_len - 1 (Causal LM shift)
                # outputs.loss 已经是这段序列有效 token 的平均 cross_entropy loss
                neg_log_likelihood = outputs.loss * (trg_len - 1)
                
            total_nll += neg_log_likelihood.item()
            total_tokens += (trg_len - 1)

    if total_tokens == 0:
        return float('nan')
        
    ppl = torch.exp(torch.tensor(total_nll / total_tokens))
    return ppl.item()

# 1. 评测 Wikitext
wiki_test = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
# 设置 join_texts=True 解决 Wikitext 句子极短和碎片化的问题
wiki_ppl = calculate_ppl(model, tokenizer, wiki_test, text_column="text", sample_size=1000, join_texts=True) 
print(f"Wikitext-2 PPL: {wiki_ppl:.2f}")

# 2. 评测 PG-19 (PG-19数据十分庞大，按需加载评测前两本小说即可)
# 彻底避开 HuggingFace 官方源和脚本限制问题，直接使用社区已经清洗转换成现代 parquet 格式的镜像
pg19_test = load_dataset("emozilla/pg19-test", split="test", streaming=True)
pg19_sample = list(pg19_test.take(10)) # 取流中的前10本进行评估
pg19_ppl = calculate_ppl(model, tokenizer, pg19_sample, text_column="text", sample_size=10, join_texts=False)
print(f"PG-19 PPL: {pg19_ppl:.2f}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Evaluating PPL: 100%|██████████| 1/1 [00:13<00:00, 13.22s/it]


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Evaluating PPL: 100%|██████████| 1/1 [00:13<00:00, 13.22s/it]


...

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Evaluating PPL: 100%|██████████| 1/1 [00:13<00:00, 13.22s/it]


Wikitext-2 PPL: 70.57
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Evaluating PPL: 100%|██████████| 10/10 [02:24<00:00, 14.46s/it]

PG-19 PPL: 53.25


In [10]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm
from streaming_llm.enable_streaming_llm import enable_streaming_llm

# 如果是 Apple Silicon (M1/M2/M3)，优先使用 mps 加速
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

model_id = "EleutherAI/pythia-70m-deduped"
# 重新加载到对应的设备上
model_streaming = AutoModelForCausalLM.from_pretrained(
  model_id,
  revision="step3000",
  cache_dir="./pythia-70m-deduped/step3000",
).to(device)
model_streaming.eval()

enable_streaming_llm(model_streaming, start_size=4, recent_size=1020)

Using device: mps


/opt/anaconda3/envs/streaming/lib/python3.8/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of GPTNeoXForCausalLM were not initialized from the model checkpoint at EleutherAI/pythia-70m-deduped and are newly initialized: ['gpt_neox.layers.5.attention.rotary_emb.inv_freq', 'gpt_neox.layers.3.attention.rotary_emb.inv_freq', 'gpt_neox.layers.2.attention.rotary_emb.inv_freq', 'gpt_neox.layers.0.attention.rotary_emb.inv_freq', 'gpt_neox.layers.4.attention.rotary_emb.inv_freq', 'gpt_neox.layers.1.attention.rotary_emb.inv_freq']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: mps


/opt/anaconda3/envs/streaming/lib/python3.8/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of GPTNeoXForCausalLM were not initialized from the model checkpoint at EleutherAI/pythia-70m-deduped and are newly initialized: ['gpt_neox.layers.5.attention.rotary_emb.inv_freq', 'gpt_neox.layers.3.attention.rotary_emb.inv_freq', 'gpt_neox.layers.2.attention.rotary_emb.inv_freq', 'gpt_neox.layers.0.attention.rotary_emb.inv_freq', 'gpt_neox.layers.4.attention.rotary_emb.inv_freq', 'gpt_neox.layers.1.attention.rotary_emb.inv_freq']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


StartRecentKVCache: 4, 1020


In [7]:
prompt = "The future of artificial intelligence in natural language processing is"
latency_metrics = measure_latency(model_streaming, tokenizer, prompt, max_new_tokens=64)
print("=== 性能指标 Streaming_LLM ===")
for k, v in latency_metrics.items():
    print(f"{k}: {v}")

=== 性能指标 Streaming_LLM ===
TTFT (ms): 437.13
TPOT (ms/token): 79.25
Throughput (tokens/s): 11.79


In [14]:
import torch
import math
from tqdm import tqdm
from datasets import load_dataset
from streaming_llm.enable_streaming_llm import enable_streaming_llm

def calculate_streaming_ppl(model, tokenizer, dataset, text_column, sample_size=1, max_tokens=4000, chunk_size=256, kv_cache_evictor=None):
    """
    真正的“流式” PPL 评测：把过去的记忆 (past_key_values) 连续往后保留！
    如果传入了 kv_cache_evictor，会在每次生成后截断缓存，这才是真正触发 StreamingLLM 内存优化的逻辑。
    """
    total_nll = 0.0
    total_tokens = 0
    
    texts = [dataset[i][text_column] for i in range(min(sample_size, len(dataset))) if dataset[i][text_column].strip()]
    text = "\n\n".join(texts)
    
    encodings = tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids[:, :max_tokens].to(model.device)
    seq_len = input_ids.size(1)
    
    past_key_values = None
    
    # 按照 256 为字块，像看书一样连续读下去，带着过去全部的记忆
    for i in tqdm(range(0, seq_len - 1, chunk_size), desc="Streaming Eval"):
        chunk_ids = input_ids[:, i : i + chunk_size]
        
        with torch.no_grad():
            outputs = model(chunk_ids, past_key_values=past_key_values, use_cache=True, labels=chunk_ids)
            
            valid_len = chunk_ids.size(1) - 1
            if valid_len > 0:
                total_nll += outputs.loss.item() * valid_len
                total_tokens += valid_len
            
            # StreamingLLM 的核心：必须手动截断超出的 KV 缓存！
            if kv_cache_evictor is not None:
                past_key_values = kv_cache_evictor(outputs.past_key_values)
            else:
                past_key_values = outputs.past_key_values

    return math.exp(total_nll / total_tokens) if total_tokens > 0 else float('nan')

print("Loading PG-19 test data...")
pg19_test = load_dataset("emozilla/pg19-test", split="test", streaming=True)
pg19_sample = list(pg19_test.take(1))

print("=== 初始化 StreamingLLM 的 KV Cache 截断器 ===")
# 之前在其他 cell 调用了 enable_streaming_llm 但没有保存返回值，
# 我们这里重新调用并保存，因为它的内部有 evict 逻辑处理。
kv_cache_evictor = enable_streaming_llm(model_streaming, start_size=4, recent_size=1020)

print("开始长文档连续流式测试 (4000 tokens)")
print("=== 无Streaming_LLM版本 ===")
pg19_ppl_baseline = calculate_streaming_ppl(model, tokenizer, pg19_sample, text_column="text", max_tokens=4000, chunk_size=256)
print(f"PG-19 PPL Baseline: {pg19_ppl_baseline:.2f}")

print("\n=== 包含Streaming_LLM版本 ===")
pg19_ppl_streaming = calculate_streaming_ppl(model_streaming, tokenizer, pg19_sample, text_column="text", max_tokens=4000, chunk_size=256, kv_cache_evictor=kv_cache_evictor)
print(f"PG-19 PPL StreamingLLM: {pg19_ppl_streaming:.2f}")

Loading PG-19 test data...
=== 初始化 StreamingLLM 的 KV Cache 截断器 ===
StartRecentKVCache: 4, 1020
开始长文档连续流式测试 (4000 tokens)
=== 无Streaming_LLM版本 ===


Streaming Eval: 100%|██████████| 16/16 [00:34<00:00,  2.15s/it]


PG-19 PPL Baseline: 68.03

=== 包含Streaming_LLM版本 ===


Streaming Eval: 100%|██████████| 16/16 [00:04<00:00,  3.72it/s]

PG-19 PPL StreamingLLM: 40.36
